# LenovoStore - Análisis de Inventario y API REST
Este notebook muestra cómo procesar archivos de inventario en Colab y cómo consumir la API de productos creada en Node.js.


## 1. Configuración del Entorno
Instalamos las librerías necesarias para análisis de datos y peticiones HTTP.


In [ ]:
!pip install requests pandas matplotlib

## 2. Gestión de Archivos en la Nube (Google Drive)
Montamos Google Drive para acceder a archivos CSV de inventario almacenados en la nube.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Procesamiento de Inventario desde CSV
Leemos un archivo `laptops.csv` (previamente subido a Drive) y realizamos un análisis básico.


In [ ]:
import pandas as pd

# Ruta al archivo en Drive
csv_path = '/content/drive/MyDrive/laptops.csv'

# En caso de no tener el archivo en Drive, podemos crear un ejemplo rápido:
try:
    df = pd.read_csv(csv_path)
except:
    print("Archivo no encontrado en Drive. Usando datos de ejemplo...")
    data = {
        'brand': ['Lenovo', 'Lenovo', 'Apple', 'MSI'],
        'model': ['ThinkPad X1', 'Legion 5', 'MacBook Pro', 'Katana GF66'],
        'price': [1899, 1200, 2400, 1100],
        'stock': [15, 10, 5, 8]
    }
    df = pd.DataFrame(data)

print(df.head())
total_value = (df['price'] * df['stock']).sum()
print(f'Valor total estimado en inventario: ${total_value:,.2f}')

## 4. Visualización de Datos
Generamos un gráfico para visualizar la distribución del stock.


In [ ]:
import matplotlib.pyplot as plt

df.groupby('brand')['stock'].sum().plot(kind='bar', color='#e1251b')
plt.title('Stock Total por Marca')
plt.ylabel('Unidades')
plt.show()

## 5. Comunicación con la API REST (Node.js)
Interactuamos con el backend para obtener y crear productos.


In [ ]:
import requests

# URL de la API (Asegúrate de que el túnel o la IP sea correcta)
API_URL = 'http://localhost:3000/api/products'

def get_products():
    try:
        response = requests.get(API_URL)
        if response.status_code == 200:
            return response.json()
        return []
    except:
        return "Error: No se pudo conectar a la API. ¿Está el servidor corriendo?"

print("Productos en la API:", get_products())

## 6. Sincronización: Cloud -> API
Ejemplo de cómo subir los datos procesados en Colab hacia nuestra API en Node.js.


In [ ]:
def sync_to_api(row):
    payload = {
        "name": row['model'],
        "brand": row['brand'],
        "category": "Laptops",
        "description": "Importado desde Colab",
        "price": float(row['price']),
        "stock": int(row['stock'])
    }
    # Descomentar para ejecutar cuando la API esté accesible:
    # requests.post(API_URL, json=payload)
    print(f"Sincronizando: {payload['name']}")

df.apply(sync_to_api, axis=1)